In [1]:
from datasets import Dataset, load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
dataset = dataset.filter(lambda x: x["category"] == "ACCOUNT")
dataset = dataset.shuffle(seed=122) # 42
# dataset[0]

In [2]:
import dspy

train_samples = dataset.select(range(1000))
dev_samples = dataset.select(range(1000, 1100))

trainset = [dspy.Example(instruction=ex['instruction'], intent=ex['intent']).with_inputs('instruction') for ex in train_samples]
devset = [dspy.Example(instruction=ex['instruction'], intent=ex['intent']).with_inputs('instruction') for ex in dev_samples]

print(f"Anzahl der Beispiele im Trainingsset: {len(trainset)}")
print(f"Anzahl der Beispiele im Evaluationsset: {len(devset)}")

# Check 5 elements of the trainset
# print(trainset[:5])

Anzahl der Beispiele im Trainingsset: 1000
Anzahl der Beispiele im Evaluationsset: 100


In [3]:
# Prüfen, ob alle Intents in train und dev vorhanden sind
train_intents = list(train_samples.unique("intent"))
print(train_intents)
dev_intents = list(dev_samples.unique("intent"))
print(dev_intents)

Flattening the indices:   0%|          | 0/1000 [00:00<?, ? examples/s]

['edit_account', 'recover_password', 'switch_account', 'delete_account', 'create_account', 'registration_problems']


Flattening the indices:   0%|          | 0/100 [00:00<?, ? examples/s]

['create_account', 'switch_account', 'edit_account', 'registration_problems', 'delete_account', 'recover_password']


In [4]:
class IntentSignature(dspy.Signature):
    """
    Identify the intent of the customer instruction. 
    Possible values are recover_password, switch_account, create_account, delete_account, registration_problems, edit_account.
    """
    instruction = dspy.InputField(desc="Instruction: a user request from the Customer Service domain.")
    intent = dspy.OutputField(desc="Intent: the intent corresponding to the user instruction.")

class IntentClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.Predict(IntentSignature)

    def forward(self, instruction):
        return self.predictor(instruction=instruction)

In [5]:
# Konfiguration des lokalen Sprachmodells
local_llm = dspy.LM(
    "openai/gemma-3-4b-it-Q4_K_M.gguf", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=0.1,
    cache=False
)

dspy.configure(lm=local_llm)

In [6]:
from dspy.evaluate import Evaluate

# Metrik-Funktion: Vergleicht Vorhersage und Label (case-insensitive)
def exact_match_metric(gold, pred, trace=None):
    return gold.intent.lower() == pred.intent.lower()

# Evaluator instanziieren
evaluator = Evaluate(devset=devset, metric=exact_match_metric, num_threads=1, display_progress=True)

In [7]:
from dspy.teleprompt import BootstrapFewShot

# 1. Evaluation des unoptimierten Modells (Zero-Shot)
print("Evaluation vor Optimierung (Zero-Shot):")
unoptimized_program = IntentClassifier()
zero_shot_score = evaluator(unoptimized_program, display_table=0)

# 2. Iterative Optimierung und Evaluation
results = {"0 (Zero-Shot)": zero_shot_score}
demo_counts = [2, 4, 8]

for count in demo_counts:
    print(f"\nStarte Optimierung mit max_bootstrapped_demos = {count}...")

    # Konfiguration des Optimizers mit der jeweiligen Beispielanzahl
    optimizer = BootstrapFewShot(metric=exact_match_metric, max_bootstrapped_demos=count)

    # Kompilierung des Programms
    optimized_program = optimizer.compile(IntentClassifier(), trainset=trainset)

    print(f"\nEvaluation nach Optimierung (Demos={count}):")
    score = evaluator(optimized_program, display_table=0)
    results[str(count)] = score

print("\n--- Experiment-Ergebnisse ---")
for demos, score in results.items():
    print(f"Anzahl Demos: {demos}, Accuracy: {score.score}%")

Evaluation vor Optimierung (Zero-Shot):
Average Metric: 75.00 / 100 (75.0%): 100%|███████████████████████████████████████████████████████████████████████████████| 100/100 [00:24<00:00,  4.14it/s]

2025/11/14 10:22:54 INFO dspy.evaluate.evaluate: Average Metric: 75 / 100 (75.0%)




Starte Optimierung mit max_bootstrapped_demos = 2...


  0%|▏                                                                                                                    | 2/1000 [00:03<26:18,  1.58s/it]


Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.

Evaluation nach Optimierung (Demos=2):
Average Metric: 99.00 / 100 (99.0%): 100%|███████████████████████████████████████████████████████████████████████████████| 100/100 [00:25<00:00,  3.93it/s]

...



Starte Optimierung mit max_bootstrapped_demos = 8...


  1%|▉                                                                                                                    | 8/1000 [00:04<09:02,  1.83it/s]


Bootstrapped 8 full traces after 8 examples for up to 1 rounds, amounting to 8 attempts.

Evaluation nach Optimierung (Demos=8):
Average Metric: 98.00 / 100 (98.0%): 100%|███████████████████████████████████████████████████████████████████████████████| 100/100 [00:25<00:00,  3.89it/s]

2025/11/14 10:24:22 INFO dspy.evaluate.evaluate: Average Metric: 98 / 100 (98.0%)




--- Experiment-Ergebnisse ---
Anzahl Demos: 0 (Zero-Shot), Accuracy: 75.0%
Anzahl Demos: 2, Accuracy: 99.0%
Anzahl Demos: 4, Accuracy: 96.0%
Anzahl Demos: 8, Accuracy: 98.0%
